### Add Agent

In [1]:
from pathlib import Path
import subprocess

parent_dir = Path.cwd().parent
proc_agent = subprocess.Popen(["node", "main.js"], cwd=str(parent_dir))
print(f"Running (PID={proc_agent.pid})")

Running (PID=45128)


In [2]:
import time
time.sleep(15)

In [3]:
# # If necessary, when something goes wrong during debugging, kill proc with pid
# import subprocess
# for pid in [43812]: # PID to force kill
#     print(subprocess.run(["taskkill", "/PID", str(pid), "/F"]))

### Construction and Evaluation

In [4]:
# --- Load prompts from the mapping (single source of truth) ---
from pathlib import Path
import json

# Load JSON files from benchmarks folder only (not archive/; archive = excluded from evaluation)
# Schema: prompts = [[turn1, turn2, ...], ...], checks = [[eval_turn1], [eval_turn2], ...]
BENCHMARKS_DIR = Path.cwd() / "benchmarks"
json_files = sorted(BENCHMARKS_DIR.glob("*.json"))
mapping = []
for json_file in json_files:
    file_mapping = json.loads(json_file.read_text(encoding="utf-8"))
    mapping.extend(file_mapping)

# Build runs: each run = (prompt_sequence, checks_per_turn)
runs = []
for item in mapping:
    for prompt_sequence in item["prompts"]:
        runs.append((prompt_sequence, item["checks"]))

print(f"[PY] Benchmarks dir: {BENCHMARKS_DIR.resolve()} (cwd: {Path.cwd().resolve()})")
if len(runs) == 0:
    print("[PY] ⚠️ No runs. Put at least one .json in benchmarks/ (not in archive/), or run notebook from the folder that contains benchmarks/.")
print("Total problem #:", len(runs))
for i, (prompts, _) in enumerate(runs):
    print(i, prompts)

[PY] Benchmarks dir: D:\git\mineCEraft\mineCEraft\benchmarks (cwd: D:\git\mineCEraft\mineCEraft)
Total problem #: 11
0 ['Build a stone arch bridge.']
1 ['Lay the foundation for a 15x20 block rectangular building. Use stone blocks and make it two blocks deep.']
2 ['Lay the foundation for a 20x15 block rectangular building. Use stone blocks and make it two blocks deep.']
3 ['Build a two-room house.', 'The client changed their mind. Revise it to a three-room house.']
4 ['Build an arch bridge.']
5 ['Build an arched bridge.']
6 ['Build an arch bridge, using only stone blocks.']
7 ['Build an arched bridge, using only stone blocks.']
8 ["Build an arch bridge, using only stone blocks. Let's think step by step."]
9 ["Build an arched bridge, using only stone blocks. Let's think step by step."]
10 ['Construct an arch bridge with a width of 2 and a span of 5 (capable of crossing a river 5 units wide). The bridge must be supported by 2x2 area supports at both ends, and the highest point of the arch

In [5]:
import subprocess, shutil, json
from pathlib import Path
from action_processor import read_placed_and_removed_from_action

# Path to Node script in current directory
script = (Path.cwd() / "send_prompts.js").resolve()
node = shutil.which("node") or "node"

proc = subprocess.Popen(
    [node, str(script)],
    cwd=str(script.parent),       # Node's process.cwd() equals the JS folder
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1,                    # line-buffered
)

# Build payload from runs (Node still expects flat prompts + run_lengths)
prompts_for_send = [p for (seq, _) in runs for p in seq]
run_lengths_for_send = [len(seq) for (seq, _) in runs]
proc.stdin.write(json.dumps({
    "prompts": prompts_for_send,
    "run_lengths": run_lengths_for_send,
    "clear_between": True,
    "inter_prompt_command": "Come up to the highest block position, and move 30 blocks in the positive z direction.",
    "inter_prompt_delay": 5000
}) + "\n")
proc.stdin.close()

SENTINEL = "::ACTION_MAX_JS::" # indicator of the lastly executed JS file names for each prompt.
# Each entry: (placed, removed) for that turn (for multi-turn we merge into cumulative coords in evaluation)
arr_of_deltas = []

try:
    for line in proc.stdout:
        line = line.rstrip("\n")
        print(line)  # always mirror Node logs

        # If Node reports the max-numbered file, parse and print it
        if line.startswith(SENTINEL):
            payload_raw = line[len(SENTINEL):]
            try:
                payload = json.loads(payload_raw)
            except json.JSONDecodeError:
                print("[PY] Failed to parse sentinel JSON.")
                continue

            if payload.get("ok") and "path" in payload:
                file_path = Path(payload["path"])
                print(f"\n[PY] Max action file: index={payload.get('index')} name={payload.get('name')}")
                print(f"[PY] Path: {file_path}")

                try:
                    placed, removed = read_placed_and_removed_from_action(str(file_path))
                    print(f"[PY] placed={len(placed)}, removed={len(removed)}")
                    arr_of_deltas.append((placed, removed))
                                       
                except Exception as e:
                    print(f"[PY] Failed to convert action to coords: {e}")
                    
            else:
                # e.g., dir not found or no js files
                reason = payload.get("reason", "unknown")
                print(f"[PY] No max file reported (reason={reason}).")
finally:
    proc.stdout.close()
    proc.wait()


ℹ️ Using agent name: builder
🧹 Cleared all files under D:\git\mineCEraft\bots\builder\action-code
✅ Connected to MindServer at http://localhost:8080 (socket id=hZrCZc3AcjM9ZqlOAAAD)

➡️ Sending to builder (run 1, turn 1/1): "Build a stone arch bridge."
⏳ Waiting for completion keyword (timeout 10 min)...
📨 [builder] I'll build a stone arch bridge right away! Let me get started on that. !newAction("Build a stone arch bridge at my current location using stone, stone bricks, and slabs for an elegant design with proper arch support")
📨 [builder] I've successfully built a stone arch bridge! The bridge spans 12 blocks with a width of 5 blocks, featuring a graceful arch design. I used stone for the structural supports, stone bricks for the bridge deck, and added stone brick walls as railings along both sides. The underside is decorated with stone slabs following the arch curve for added detail and aesthetic appeal. 
✅ Completion detected for "Build a stone arch bridge.".
::ACTION_MAX_JS::{"ok

In [6]:
import json
import importlib
from pathlib import Path
from collections import defaultdict

# Filled when evaluation runs (matching arr_of_deltas); use in Visualization cell: coords_by_problem[problem_i][turn_j]
coords_by_problem = []

# Use runs from Load cell; only load mapping/runs if running this cell standalone (runs not defined)
try:
    runs
except NameError:
    BENCHMARKS_DIR = Path.cwd() / "benchmarks"
    mapping = []
    for json_file in sorted(BENCHMARKS_DIR.glob("*.json")):
        file_mapping = json.loads(json_file.read_text(encoding="utf-8"))
        mapping.extend(file_mapping)
    runs = []
    for item in mapping:
        for prompt_sequence in item["prompts"]:
            runs.append((prompt_sequence, item["checks"]))

def resolve_callable(dotted: str):
    """Resolve e.g. 'size.is_equal' to eval_code.size.is_equal callable."""
    fq = f"eval_code.{dotted}"
    mod_name, func_name = fq.rsplit(".", 1)
    mod = importlib.import_module(mod_name)
    return getattr(mod, func_name), fq

def run_checks(coords, checks):
    """Run checks on coords; return score, results, and by_category summary."""
    results = []
    score = 0
    cat_pass = defaultdict(int)   # category -> passed count
    cat_total = defaultdict(int)  # category -> total count

    for chk in checks:
        fn_name = chk["fn"]              # e.g., 'material.is_quantity_correct'
        args = chk.get("args") or {}
        category = fn_name.split(".", 1)[0]  # module name as category (e.g, material)

        fn, fq = resolve_callable(fn_name)
        try:
            ok = 1 if bool(fn(coords, **args)) else 0
        except Exception as e:
            ok, args = 0, {**args, "_error": str(e)}  # surface error on this line

        score += ok
        cat_total[category] += 1
        cat_pass[category]  += ok

        results.append({"fn": fq, "category": category, "ok": ok, "args": args})

    # shape as plain dicts for printing
    cat_summary = {
        c: {"pass": cat_pass[c], "total": cat_total[c]}
        for c in sorted(cat_total.keys())
    }
    return {"score": score, "total": len(results), "results": results, "by_category": cat_summary}

overall_pass = 0
overall_total = 0
overall_by_cat_pass  = defaultdict(int)
overall_by_cat_total = defaultdict(int)

def merge_coords(prev: list, placed: list, removed: list) -> list:
    """Return prev with removed positions dropped and placed added (multi-turn cumulative)."""
    rem_set = {(c["x"], c["y"], c["z"]) for c in removed}
    prev_remaining = [c for c in prev if (c["x"], c["y"], c["z"]) not in rem_set]
    by_key = {(c["x"], c["y"], c["z"]): c for c in prev_remaining}
    for c in placed:
        by_key[(c["x"], c["y"], c["z"])] = c
    return list(by_key.values())

total_turns = sum(len(seq) for (seq, _) in runs)
if len(arr_of_deltas) != total_turns:
    print(f"[PY] Skipping evaluation: arr_of_deltas has {len(arr_of_deltas)} entries but runs have {total_turns} turns.")
    print("[PY] Run the construction cell (send prompts) first, wait for all prompts to complete, then run this evaluation cell.")
if len(arr_of_deltas) == total_turns:
    coord_index = 0
    coords_by_problem = []
    for run_idx, (prompt_sequence, checks_per_turn) in enumerate(runs):
        n_turns = len(prompt_sequence)
        cumulative = []
        coords_this_problem = []
        for turn_idx in range(n_turns):
            placed, removed = arr_of_deltas[coord_index]
            coord_index += 1
            cumulative = merge_coords(cumulative, placed, removed)
            prompt_text = prompt_sequence[turn_idx]
            coords = cumulative
            checks_this_turn = checks_per_turn[turn_idx]
            
            report = run_checks(coords, checks_this_turn)
            coords_this_problem.append(cumulative)
            
            # --- Pretty print per-turn report ---
            print("\n[PY] === Evaluation Result ===")
            print(f"[PY] Run #{run_idx+1}, Turn #{turn_idx+1}/{n_turns}: {prompt_text}")
            print(f"[PY] Score: {report['score']} / {report['total']} (coords={len(coords)})")
            
            # Category breakdown for this turn
            print("[PY] Category scores:")
            for cat, st in report["by_category"].items():
                print(f"  - {cat}: {st['pass']} / {st['total']}")
            
            # Individual checks
            for r in report["results"]:
                status = "PASS" if r["ok"] else "FAIL"
                line = f"  · {status} | {r['fn']}({r.get('args', {})})"
                print(line)
            
            # Accumulate overall totals
            overall_pass  += report["score"]
            overall_total += report["total"]
            for cat, st in report["by_category"].items():
                overall_by_cat_pass[cat]  += st["pass"]
                overall_by_cat_total[cat] += st["total"]
        coords_by_problem.append(coords_this_problem)

    # --- Overall summary across all prompts ---
    if overall_total > 0:
        print("\n[PY] === Overall Summary ===")
        overall_pct = (overall_pass / overall_total * 100) if overall_total > 0 else 0.0
        print(f"[PY] Total PASS: {overall_pass} / {overall_total} ({overall_pct:.1f}%)")
        print("[PY] Category totals:")
        for cat in sorted(overall_by_cat_total.keys()):
            p = overall_by_cat_pass[cat]
            t = overall_by_cat_total[cat]
            cat_pct = (p / t * 100) if t > 0 else 0.0
            print(f"  - {cat}: {p} / {t} ({cat_pct:.1f}%)")

    if len(coords_by_problem) == 0:
        print("[PY] ⚠️ coords_by_problem is empty. Run Load → Construction → Evaluation (benchmarks/ must have .json files).")



[PY] === Evaluation Result ===
[PY] Run #11, Turn #1/1: Construct an arch bridge with a width of 2 and a span of 5 (capable of crossing a river 5 units wide). The bridge must be supported by 2x2 area supports at both ends, and the highest point of the arch must reach a height of 4.
[PY] Score: 2 / 4 (coords=50)
[PY] Category scores:
  - shape: 0 / 2
  - structural_stability: 2 / 2
  · PASS | eval_code.structural_stability.is_ground_connected({})
  · FAIL | eval_code.shape.is_top_surface_concave({'strict': 'non-flat'})
  · FAIL | eval_code.shape.is_bottom_surface_concave({'strict': 'non-flat'})
  · PASS | eval_code.structural_stability.is_stress_safe({'von_mises_stress': 5337})

[PY] === Overall Summary ===
[PY] Total PASS: 23 / 43 (53.5%)
[PY] Category totals:
  - material: 8 / 8 (100.0%)
  - planning: 0 / 2 (0.0%)
  - shape: 2 / 18 (11.1%)
  - size: 2 / 2 (100.0%)
  - structural_stability: 11 / 13 (84.6%)


### Remove Agent

In [7]:
proc_agent.kill()
print("Process terminated.")

Process terminated.


### Double-check & Visualization

In [8]:
# coords_by_problem[problem_index][turn_index] = coords at that eval (problem = load cell print index).
from eval_code.viz import plotly_blocks

problem_index = 0   # problem number from load cell (print(i, prompts))
turn_index = -1     # 0=first turn, -1=last turn for that problem

try:
    coords_by_problem
except NameError:
    print("Run the evaluation cell first so coords_by_problem is available.")
else:
    n = len(coords_by_problem)
    if n == 0:
        print("No problems in coords_by_problem. Run Load → Construction → Evaluation (with matching arr_of_deltas), then try again.")
    elif problem_index < 0 or problem_index >= n:
        print(f"problem_index must be 0..{n-1} (there are {n} problems). Current problem_index={problem_index}")
    else:
        turns = coords_by_problem[problem_index]
        if turns:
            plotly_blocks.plot(turns[turn_index])
        else:
            print(f"Problem {problem_index} has no turns/coords.")
            
runs[problem_index]

(['Build a stone arch bridge.'],
 [[{'fn': 'structural_stability.is_ground_connected', 'args': {}},
   {'fn': 'shape.is_top_surface_concave', 'args': {'strict': 'non-flat'}},
   {'fn': 'shape.is_bottom_surface_concave', 'args': {'strict': 'non-flat'}}]])